In [1]:
# =========================
# Cell 1: Environment Check
# =========================

import sys
import pandas as pd
import numpy as np
import csv
import sys
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix


print("Python executable:")
print(sys.executable)
print("Python version:")
print(sys.version)

print("\nLibraries available:")
print("pandas:", pd.__version__)
print("numpy :", np.__version__)

print("\n✅ Environment ready for CSV-based analysis")
print("⚠ Plotting libraries intentionally skipped")

Python executable:
/usr/local/src/miniconda/py37-16/bin/python
Python version:
3.7.16 (default, Jan 17 2023, 22:20:44) 
[GCC 11.2.0]

Libraries available:
pandas: 1.3.5
numpy : 1.21.5

✅ Environment ready for CSV-based analysis
⚠ Plotting libraries intentionally skipped


In [2]:
# =========================
# Cell 2: Load HR Data (CSV)
# =========================

# Increase CSV field size limit (for large HR exports)
csv.field_size_limit(sys.maxsize)

DATA_FILE = "HR_Analytics.csv"

print("Attempting to load:", DATA_FILE)

# Check file exists
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        "❌ HR_Analytics_clean.csv not found. "
        "Place it in the same folder as the notebook."
    )

# Robust CSV load with auto delimiter detection
df = pd.read_csv(
    DATA_FILE,
    encoding="latin1",
    sep=None,
    engine="python",
    skip_blank_lines=True
)

print("\n✅ CSV loaded successfully")
print("Shape:", df.shape)

print("\nColumns detected:")
for col in df.columns:
    print(" -", col)

df.head()

Attempting to load: HR_Analytics_clean.csv

✅ CSV loaded successfully
Shape: (1470, 44)

Columns detected:
 - Attrition
 - Business Travel
 - CF_age band
 - CF_attrition label
 - Department
 - Education Field
 - emp no
 - Employee Number
 - Gender
 - Job Role
 - Marital Status
 - Over Time
 - Over18
 - Training Times Last Year
 - -2
 - 0
 - Age
 - CF_attrition count
 - CF_attrition counts
 - CF_attrition rate
 - CF_current Employee
 - Daily Rate
 - Distance From Home
 - Education
 - Employee Count
 - Environment Satisfaction
 - Hourly Rate
 - Job Involvement
 - Job Level
 - Job Satisfaction
 - Monthly Income
 - Monthly Rate
 - Num Companies Worked
 - Percent Salary Hike
 - Performance Rating
 - Relationship Satisfaction
 - Standard Hours
 - Stock Option Level
 - Total Working Years
 - Work Life Balance
 - Years At Company
 - Years In Current Role
 - Years Since Last Promotion
 - Years With Curr Manager


,Attrition,Business Travel,CF_age band,CF_attrition label,Department,Education Field,emp no,Employee Number,Gender,Job Role,...,Performance Rating,Relationship Satisfaction,Standard Hours,Stock Option Level,Total Working Years,Work Life Balance,Years At Company,Years In Current Role,Years Since Last Promotion,Years With Curr Manager
0,Yes,Travel_Rarely,35 - 44,Ex-Employees,Sales,Life Sciences,STAFF-1,1,Female,Sales Executive,...,3,1,80,0,8,1,6,4,0,5
1,No,Travel_Frequently,45 - 54,Current Employees,R&D,Life Sciences,STAFF-2,2,Male,Research Scientist,...,4,4,80,1,10,3,10,7,1,7
2,Yes,Travel_Rarely,35 - 44,Ex-Employees,R&D,Other,STAFF-4,4,Male,Laboratory Technician,...,3,2,80,0,7,3,0,0,0,0
3,No,Travel_Frequently,25 - 34,Current Employees,R&D,Life Sciences,STAFF-5,5,Female,Research Scientist,...,3,3,80,0,8,3,8,7,3,0
4,No,Travel_Rarely,25 - 34,Current Employees,R&D,Medical,STAFF-7,7,Male,Laboratory Technician,...,3,4,80,1,6,3,2,2,2,2


In [3]:
# =========================
# Cell 3: Data Sanity Checks (Original Columns)
# =========================

print("Initial dataset shape:", df.shape)

# ---- List available columns ----
print("\nColumns available in dataset:")
for col in df.columns:
    print(" -", col)

# ---- Explicit column mapping (based on YOUR dataset) ----
COL_ATTRITION = "Attrition"
COL_DEPARTMENT = "Department"
COL_GENDER = "Gender"
COL_MONTHLY_INCOME = "Monthly Income"
COL_YEARS_AT_COMPANY = "Years At Company"
COL_EMP_ID = "Employee Number" if "Employee Number" in df.columns else "emp no"

required_columns = [
    COL_ATTRITION,
    COL_DEPARTMENT,
    COL_GENDER,
    COL_MONTHLY_INCOME,
    COL_YEARS_AT_COMPANY
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(
        f"❌ Missing required columns: {missing}\n"
        "Dataset structure does not match expected HR layout."
    )

print("\n✅ All required columns are present")

# ---- Attrition distribution ----
print("\nAttrition value counts:")
print(df[COL_ATTRITION].value_counts(dropna=False))

# ---- Basic missing value check (top 10) ----
print("\nMissing values per column (top 10):")
print(df.isna().sum().sort_values(ascending=False).head(10))

# ---- Preview data ----
df.head()

Initial dataset shape: (1470, 44)

Columns available in dataset:
 - Attrition
 - Business Travel
 - CF_age band
 - CF_attrition label
 - Department
 - Education Field
 - emp no
 - Employee Number
 - Gender
 - Job Role
 - Marital Status
 - Over Time
 - Over18
 - Training Times Last Year
 - -2
 - 0
 - Age
 - CF_attrition count
 - CF_attrition counts
 - CF_attrition rate
 - CF_current Employee
 - Daily Rate
 - Distance From Home
 - Education
 - Employee Count
 - Environment Satisfaction
 - Hourly Rate
 - Job Involvement
 - Job Level
 - Job Satisfaction
 - Monthly Income
 - Monthly Rate
 - Num Companies Worked
 - Percent Salary Hike
 - Performance Rating
 - Relationship Satisfaction
 - Standard Hours
 - Stock Option Level
 - Total Working Years
 - Work Life Balance
 - Years At Company
 - Years In Current Role
 - Years Since Last Promotion
 - Years With Curr Manager

✅ All required columns are present

Attrition value counts:
No     1233
Yes     237
Name: Attrition, dtype: int64

Missing va

,Attrition,Business Travel,CF_age band,CF_attrition label,Department,Education Field,emp no,Employee Number,Gender,Job Role,...,Performance Rating,Relationship Satisfaction,Standard Hours,Stock Option Level,Total Working Years,Work Life Balance,Years At Company,Years In Current Role,Years Since Last Promotion,Years With Curr Manager
0,Yes,Travel_Rarely,35 - 44,Ex-Employees,Sales,Life Sciences,STAFF-1,1,Female,Sales Executive,...,3,1,80,0,8,1,6,4,0,5
1,No,Travel_Frequently,45 - 54,Current Employees,R&D,Life Sciences,STAFF-2,2,Male,Research Scientist,...,4,4,80,1,10,3,10,7,1,7
2,Yes,Travel_Rarely,35 - 44,Ex-Employees,R&D,Other,STAFF-4,4,Male,Laboratory Technician,...,3,2,80,0,7,3,0,0,0,0
3,No,Travel_Frequently,25 - 34,Current Employees,R&D,Life Sciences,STAFF-5,5,Female,Research Scientist,...,3,3,80,0,8,3,8,7,3,0
4,No,Travel_Rarely,25 - 34,Current Employees,R&D,Medical,STAFF-7,7,Male,Laboratory Technician,...,3,4,80,1,6,3,2,2,2,2


In [4]:
# =========================
# Cell 4: HR KPIs (Using CF Values)
# =========================

# ---- Column references (exact names from your dataset) ----
COL_DEPARTMENT = "Department"
COL_GENDER = "Gender"
COL_ATTRITION = "Attrition"
COL_CF_ATTRITION_RATE = "CF_attrition rate"
COL_CF_ATTRITION_COUNT = "CF_attrition count"

# ---- Headcount ----
headcount = len(df)

# ---- Overall Attrition Rate (from Attrition column – this is fine) ----
overall_attrition_rate = df[COL_ATTRITION].eq("Yes").mean() * 100

print(f"Total Headcount: {headcount}")
print(f"Overall Attrition Rate: {overall_attrition_rate:.2f}%\n")

# ---- Department-wise Attrition (USING CF ATTRITION RATE) ----
attrition_by_department = (
    df.groupby(COL_DEPARTMENT)[COL_CF_ATTRITION_RATE]
    .mean()
    .round(2)
    .sort_values(ascending=False)
)

print("✅ Attrition Rate by Department (CF-based):")
display(attrition_by_department)

# ---- Gender-wise Attrition (still valid to calculate directly) ----
attrition_by_gender = (
    df.groupby(COL_GENDER)[COL_ATTRITION]
    .apply(lambda s: s.eq("Yes").mean() * 100)
    .round(2)
)

print("\n✅ Attrition Rate by Gender:")
display(attrition_by_gender)

Total Headcount: 1470
Overall Attrition Rate: 16.12%

✅ Attrition Rate by Department (CF-based):


Department
Sales    0.21
HR       0.19
R&D      0.14
Name: CF_attrition rate, dtype: float64


✅ Attrition Rate by Gender:


Gender
Female    14.80
Male      17.01
Name: Attrition, dtype: float64

In [5]:
# =========================
# Cell 5: Save KPIs to JSON
# =========================

import json
import os

# ---- Output folder ----
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Prepare KPI dictionary ----
kpi_results = {
    "headcount": int(headcount),
    "overall_attrition_rate_pct": round(float(overall_attrition_rate), 2),
    "attrition_by_department_cf_pct": {
        dept: round(float(rate), 2)
        for dept, rate in attrition_by_department.items()
    },
    "attrition_by_gender_pct": {
        gender: round(float(rate), 2)
        for gender, rate in attrition_by_gender.items()
    }
}

# ---- Save to JSON ----
output_file = os.path.join(OUTPUT_DIR, "hr_kpi_summary.json")

with open(output_file, "w") as f:
    json.dump(kpi_results, f, indent=2)

print("✅ KPI JSON saved successfully")
print("File location:", output_file)

# Preview saved content
kpi_results

✅ KPI JSON saved successfully
File location: outputs/hr_kpi_summary.json


{'headcount': 1470,
 'overall_attrition_rate_pct': 16.12,
 'attrition_by_department_cf_pct': {'Sales': 0.21, 'HR': 0.19, 'R&D': 0.14},
 'attrition_by_gender_pct': {'Female': 14.8, 'Male': 17.01}}

In [6]:
# =========================
# Cell 6: Attrition Prediction Model
# =========================



# ---- Target variable ----
TARGET = "Attrition"

# ---- Columns to EXCLUDE (IDs + CF fields) ----
exclude_cols = [
    "emp no",
    "Employee Number",
    "CF_age band",
    "CF_attrition label",
    "CF_attrition count",
    "CF_attrition counts",
    "CF_attrition rate",
    "CF_current Employee"
]

exclude_cols = [c for c in exclude_cols if c in df.columns]

# ---- Feature matrix and target ----
X = df.drop(columns=[TARGET] + exclude_cols)
y = df[TARGET].map({"Yes": 1, "No": 0})

print("Features used:", X.shape[1])
print("Target distribution:")
print(y.value_counts())

# ---- Identify numeric and categorical columns ----
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("\nNumeric features:", len(num_cols))
print("Categorical features:", len(cat_cols))

# ---- Preprocessing pipeline ----
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

# ---- Logistic Regression model ----
model = Pipeline(steps=[
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000))
])

# ---- Train/Test Split ----
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# ---- Train model ----
model.fit(X_train, y_train)

# ---- Predictions ----
y_pred = model.predict(X_test)

# ---- Evaluation ----
print("\n✅ Classification Report:")
print(classification_report(y_test, y_pred))

print("✅ Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


print("✅ Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Features used: 35
Target distribution:
0    1233
1     237
Name: Attrition, dtype: int64

Numeric features: 26
Categorical features: 9

✅ Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.99      0.93       309
           1       0.81      0.22      0.35        59

    accuracy                           0.87       368
   macro avg       0.84      0.61      0.64       368
weighted avg       0.86      0.87      0.83       368

✅ Confusion Matrix:
[[306   3]
 [ 46  13]]
✅ Confusion Matrix:
[[306   3]
 [ 46  13]]


/usr/local/src/miniconda/py37-16/lib/python3.7/site-packages/sklearn/linear_model/_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,


In [7]:
# =========================
# Cell 7: Feature Importance
# =========================

# ---- Access trained components ----
prep = model.named_steps["prep"]
clf = model.named_steps["clf"]

# ---- Get feature names ----
num_features = prep.transformers_[0][2]  # numeric columns
cat_transformer = prep.transformers_[1][1]
cat_features = cat_transformer.get_feature_names_out(
    prep.transformers_[1][2]
)

feature_names = list(num_features) + list(cat_features)

# ---- Extract coefficients ----
coefficients = clf.coef_[0]

feature_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefficients,
        "absolute_impact": np.abs(coefficients)
    })
    .sort_values("absolute_impact", ascending=False)
)

# ---- Top 20 most important drivers ----
top_features = feature_importance.head(20)

print("✅ Top 20 Drivers of Attrition")
top_features

✅ Top 20 Drivers of Attrition


,feature,coefficient,absolute_impact
53,Over Time_Yes,0.395496,0.395496
52,Over Time_No,-0.394919,0.394919
7,Environment Satisfaction,-0.303826,0.303826
19,Stock Option Level,-0.301280,0.301280
24,Years Since Last Promotion,0.274401,0.274401
11,Job Satisfaction,-0.269672,0.269672
9,Job Involvement,-0.261439,0.261439
23,Years In Current Role,-0.232925,0.232925
51,Marital Status_Single,0.204747,0.204747
17,Relationship Satisfaction,-0.192471,0.192471


In [8]:
# =========================
# Cell 8: Export Feature Importance
# =========================

import os

# ---- Output directory ----
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- File path ----
output_file = os.path.join(OUTPUT_DIR, "attrition_feature_importance.csv")

# ---- Save top drivers ----
top_features.to_csv(output_file, index=False)

print("✅ Feature importance saved successfully")
print("File location:", output_file)

top_features

✅ Feature importance saved successfully
File location: outputs/attrition_feature_importance.csv


,feature,coefficient,absolute_impact
53,Over Time_Yes,0.395496,0.395496
52,Over Time_No,-0.394919,0.394919
7,Environment Satisfaction,-0.303826,0.303826
19,Stock Option Level,-0.301280,0.301280
24,Years Since Last Promotion,0.274401,0.274401
11,Job Satisfaction,-0.269672,0.269672
9,Job Involvement,-0.261439,0.261439
23,Years In Current Role,-0.232925,0.232925
51,Marital Status_Single,0.204747,0.204747
17,Relationship Satisfaction,-0.192471,0.192471


In [9]:
# =========================
# Cell 9: Improve Recall & Convergence
# =========================

# ---- Updated preprocessor with scaling ----
improved_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

# ---- Improved Logistic Regression ----
improved_model = Pipeline(steps=[
    ("prep", improved_preprocessor),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced"
    ))
])

# ---- Train improved model ----
improved_model.fit(X_train, y_train)

# ---- Predictions ----
y_pred_improved = improved_model.predict(X_test)

# ---- Evaluation ----
print("✅ Improved Classification Report:")
print(classification_report(y_test, y_pred_improved))

print("✅ Improved Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_improved))

✅ Improved Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.79      0.85       309
           1       0.39      0.71      0.50        59

    accuracy                           0.77       368
   macro avg       0.66      0.75      0.68       368
weighted avg       0.85      0.77      0.80       368

✅ Improved Confusion Matrix:
[[243  66]
 [ 17  42]]


In [10]:
# =========================
# Cell 10: Employee Attrition Risk Scores
# =========================

# ---- Predict attrition probabilities using improved model ----
attrition_probabilities = improved_model.predict_proba(X)[:, 1]

# ---- Create results dataframe ----
risk_df = df.copy()

risk_df["attrition_risk_score"] = attrition_probabilities
risk_df["attrition_risk_flag"] = (risk_df["attrition_risk_score"] >= 0.5).map(
    {True: "High Risk", False: "Lower Risk"}
)

# ---- Sort by risk ----
risk_df_sorted = risk_df.sort_values(
    by="attrition_risk_score", ascending=False
)

print("✅ Attrition risk scores generated")

# ---- Show top 15 highest-risk employees ----
risk_df_sorted[
    [
        "Employee Number",
        "Department",
        "Job Role",
        "Monthly Income",
        "Years At Company",
        "attrition_risk_score",
        "attrition_risk_flag"
    ]
].head(15)

✅ Attrition risk scores generated


,Employee Number,Department,Job Role,Monthly Income,Years At Company,attrition_risk_score,attrition_risk_flag
911,1273,Sales,Sales Representative,1118,1,0.994406,High Risk
463,622,R&D,Laboratory Technician,2340,1,0.992321,High Risk
1446,1494,R&D,Laboratory Technician,3172,0,0.992318,High Risk
656,911,R&D,Laboratory Technician,2795,1,0.985115,High Risk
457,614,Sales,Sales Representative,1878,0,0.981761,High Risk
14,19,R&D,Laboratory Technician,2028,4,0.977199,High Risk
357,478,Sales,Sales Representative,2174,3,0.975693,High Risk
688,959,Sales,Sales Representative,2121,1,0.973109,High Risk
1438,1624,Sales,Sales Representative,1569,0,0.967485,High Risk
1411,1504,R&D,Laboratory Technician,2561,0,0.967239,High Risk


In [11]:
# =========================
# Cell 11: Export Attrition Risk (5 Levels, Model Features Only)
# =========================

import os
import pandas as pd

# ---- Output directory ----
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Define risk level logic ----
def risk_level(score: float) -> str:
    if score >= 0.85:
        return "Very High Risk (Strong resemblance to past leavers)"
    elif score >= 0.70:
        return "High Risk"
    elif score >= 0.50:
        return "Moderate Risk"
    elif score >= 0.30:
        return "Low Risk"
    else:
        return "Very Low Risk (Strong resemblance to past stayers)"

# ---- Columns to keep: Employee info ----
employee_info_cols = [
    "Employee Number",
    "Department",
    "Job Role",
    "Gender",
    "Age"
]

employee_info_cols = [c for c in employee_info_cols if c in df.columns]

# ---- Model feature columns (only what was used) ----
model_feature_cols = num_cols + cat_cols

# Ensure they exist in dataframe
model_feature_cols = [c for c in model_feature_cols if c in df.columns]

# ---- Build final export dataframe ----
export_df = df[
    employee_info_cols + model_feature_cols
].copy()

# ---- Add prediction outputs ----
export_df["attrition_risk_score"] = attrition_probabilities
export_df["attrition_risk_level"] = export_df["attrition_risk_score"].apply(risk_level)

# ---- Sort from highest risk to lowest ----
export_df = export_df.sort_values(
    by="attrition_risk_score", ascending=False
)

# ---- Export to CSV ----
output_file = os.path.join(
    OUTPUT_DIR, "employee_attrition_risk_5_levels.csv"
)

export_df.to_csv(output_file, index=False)

print("✅ Employee attrition risk file exported successfully")
print("File location:", output_file)
print("Total employees exported:", export_df.shape[0])

# ---- Preview ----
export_df.head(10)

✅ Employee attrition risk file exported successfully
File location: outputs/employee_attrition_risk_5_levels.csv
Total employees exported: 1470


,Employee Number,Department,Job Role,Gender,Age,Training Times Last Year,-2,0,Age,Daily Rate,...,Department,Education Field,Gender,Job Role,Marital Status,Over Time,Over18,Education,attrition_risk_score,attrition_risk_level
911,1273,Sales,Sales Representative,Male,25,4,-2,0,25,599,...,Sales,Life Sciences,Male,Sales Representative,Single,Yes,Y,High School,0.994406,Very High Risk (Strong resemblance to past lea...
463,622,R&D,Laboratory Technician,Male,26,3,-2,0,26,471,...,R&D,Technical Degree,Male,Laboratory Technician,Single,Yes,Y,Bachelor's Degree,0.992321,Very High Risk (Strong resemblance to past lea...
1446,1494,R&D,Laboratory Technician,Male,24,2,-2,0,24,381,...,R&D,Medical,Male,Laboratory Technician,Single,Yes,Y,Bachelor's Degree,0.992318,Very High Risk (Strong resemblance to past lea...
656,911,R&D,Laboratory Technician,Male,32,2,-2,0,32,374,...,R&D,Life Sciences,Male,Laboratory Technician,Single,Yes,Y,Master's Degree,0.985115,Very High Risk (Strong resemblance to past lea...
457,614,Sales,Sales Representative,Male,18,3,-2,0,18,1306,...,Sales,Marketing,Male,Sales Representative,Single,Yes,Y,Bachelor's Degree,0.981761,Very High Risk (Strong resemblance to past lea...
14,19,R&D,Laboratory Technician,Male,28,4,-2,0,28,103,...,R&D,Life Sciences,Male,Laboratory Technician,Single,Yes,Y,Bachelor's Degree,0.977199,Very High Risk (Strong resemblance to past lea...
357,478,Sales,Sales Representative,Female,21,3,-2,0,21,756,...,Sales,Technical Degree,Female,Sales Representative,Single,Yes,Y,High School,0.975693,Very High Risk (Strong resemblance to past lea...
688,959,Sales,Sales Representative,Male,19,3,-2,0,19,419,...,Sales,Other,Male,Sales Representative,Single,Yes,Y,Bachelor's Degree,0.973109,Very High Risk (Strong resemblance to past lea...
1438,1624,Sales,Sales Representative,Female,18,2,-2,0,18,544,...,Sales,Medical,Female,Sales Representative,Single,Yes,Y,Associates Degree,0.967485,Very High Risk (Strong resemblance to past lea...
1411,1504,R&D,Laboratory Technician,Male,28,2,-2,0,28,289,...,R&D,Medical,Male,Laboratory Technician,Single,No,Y,Associates Degree,0.967239,Very High Risk (Strong resemblance to past lea...


In [12]:
# =========================
# Cell 12: Visual Insights (Matplotlib-Free)
# =========================

print("✅ Generating visualization-ready summaries (no matplotlib)\n")

# ---------------------------------------------------------
# 1. Distribution of Attrition Risk Scores (Quantiles)
# ---------------------------------------------------------
risk_distribution = export_df["attrition_risk_score"].describe(percentiles=[
    0.1, 0.25, 0.5, 0.75, 0.9
])

print("📊 Attrition Risk Score Distribution:")
display(risk_distribution)

# ---------------------------------------------------------
# 2. Employee Count by Risk Level (Key Intervention View)
# ---------------------------------------------------------
risk_level_counts = (
    export_df["attrition_risk_level"]
    .value_counts()
    .reset_index()
    .rename(columns={
        "index": "Risk Level",
        "attrition_risk_level": "Employee Count"
    })
)

print("\n📊 Employee Count by Attrition Risk Level:")
display(risk_level_counts)

✅ Generating visualization-ready summaries (no matplotlib)

📊 Attrition Risk Score Distribution:


count    1470.000000
mean        0.358848
std         0.283092
min         0.000555
10%         0.038497
25%         0.109315
50%         0.284608
75%         0.573497
90%         0.805575
max         0.994406
Name: attrition_risk_score, dtype: float64


📊 Employee Count by Attrition Risk Level:


,Risk Level,Employee Count
0,Very Low Risk (Strong resemblance to past stay...,759
1,Low Risk,250
2,Moderate Risk,218
3,High Risk,145
4,Very High Risk (Strong resemblance to past lea...,98


ValueError: Grouper for 'Department' not 1-dimensional